# 🧠 NeuroSync Playground

**Neural Cryptography — Train, Encrypt, Experiment**

This interactive notebook lets you explore [NeuroSync](https://github.com/CooDiiNgg/NeuroSync), a neural cryptography library where three neural networks learn to encrypt and decrypt messages through adversarial training.

| Section | What You'll Do |
|---|---|
| **1. Train a Mini Cipher** | Train a scaled-down Alice/Bob/Eve system from scratch |
| **2. Encrypt & Decrypt** | Feed your own text through the trained cipher |
| **3. Code Playground** | Write custom code against the full NeuroSync API |
| **4. Experimental Lab** | Tweak adversarial weights, key rotation, Eve strength |
| **5. Under the Hood** | Inspect network weights, temperatures, architecture |
| **6. Security Analysis** | Run the full security suite on your trained cipher |

> ⚠️ **Note**: Training parameters are intentionally scaled down so everything runs in minutes on free-tier hardware (CPU or single GPU). For production-quality results, train with the default `TrainingConfig` on a GPU for several hours.

---

## ⚙️ Setup & Installation

In [ ]:
# Install NeuroSync from PyPI
!pip install NeuroSync==0.1.6 -q

In [ ]:
import torch
import numpy as np
import time, os, sys

# Verify installation
import NeuroSync
print(f"NeuroSync v{NeuroSync.__version__} installed successfully!")
print(f"PyTorch v{torch.__version__}")
print(f"Device: {'CUDA — ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

---
## 1️⃣ Train a Mini Neural Cipher from Scratch

We'll train the full NeuroSync adversarial system — Alice (encoder), Bob (decoder), and Eve (eavesdropper) — but with **heavily scaled-down parameters** so it finishes in a few minutes.

**What's happening under the hood:**
1. Alice takes plaintext ⊕ key and produces ciphertext
2. Bob takes ciphertext ⊕ key and recovers the plaintext
3. Eve tries to crack the ciphertext *without* the key
4. Alice & Bob are jointly optimized to minimize Bob's error while maximizing Eve's error
5. Curriculum learning, maintenance mode, and dynamic adversarial scheduling keep training stable

In [ ]:
from NeuroSync import NeuroSyncTrainer, TrainingConfig

# ──────────────────────────────────────────────
# PLAYGROUND CONFIG — feel free to tweak these!
# ──────────────────────────────────────────────
playground_config = TrainingConfig(
    # Scale down for quick demo
    training_episodes=100_000,     # default: 20,000,000
    batch_size=64,
    hidden_size=512,
    num_residual_blocks=2,         # default: 3
    dropout=0.05,

    # Faster learning for small scale
    learning_rate=0.001,           # default: 0.0005
    eve_learning_rate=0.002,       # default: 0.001
    weight_decay=1e-4,

    # Adversarial training
    adversarial_max=0.15,
    eve_train_iterations=2,        # default: 3

    # Maintenance mode (auto-pause training when accuracy is high)
    maintenance_threshold=98.0,    # default: 99.0
    maintenance_threshold_exit=90.0,
    consecutive_accuracy_required=2,

    # Logging — more frequent for demo
    log_interval=1000,             # default: 2500
    test_interval=5000,            # default: 10000

    # Scheduler
    scheduler_step_size=10000,     # default: 50000
    scheduler_gamma=0.7,           # default: 0.5

    # Data — small word list generated on the fly
    data_dir="./neurosync_playground_data",
    word_list_file="words.txt",
    word_list_size=4_000,         # default: 1,000,000
)

print("Training configuration:")
for k, v in playground_config.to_dict().items():
    print(f"  {k}: {v}")

In [ ]:
# Train!
trainer = NeuroSyncTrainer(playground_config)
result = trainer.train()

print(f"\n{'='*60}")
print(f"Training complete!")
print(f"  Best accuracy: {result.best_accuracy:.1f}%")
print(f"  Final running accuracy: {result.final_accuracy:.1f}%")
print(f"{'='*60}")

In [ ]:
# Save the trained models for use in later sections
SAVE_DIR = "./neurosync_playground_weights"
result.save(SAVE_DIR)

# Also save the key so we can reload everything
trainer.key_manager.save(os.path.join(SAVE_DIR, "key.npy"))
print(f"Models saved to {SAVE_DIR}/")
print(f"  alice.pth  ({os.path.getsize(os.path.join(SAVE_DIR, 'alice.pth'))/1024:.0f} KB)")
print(f"  bob.pth    ({os.path.getsize(os.path.join(SAVE_DIR, 'bob.pth'))/1024:.0f} KB)")
print(f"  eve.pth    ({os.path.getsize(os.path.join(SAVE_DIR, 'eve.pth'))/1024:.0f} KB)")
print(f"  key.npy    ({os.path.getsize(os.path.join(SAVE_DIR, 'key.npy'))/1024:.1f} KB)")

---
## 2️⃣ Encrypt & Decrypt Your Own Messages

Now that we have trained networks, let's use the high-level `NeuroSync` cipher interface to encrypt and decrypt arbitrary text.

The system:
1. Chunks your text into 16-character blocks
2. Encodes each character as 6 bits (±1 representation)
3. XORs with the shared key → feeds through Alice → produces ciphertext
4. For decryption: XORs ciphertext with key → feeds through Bob → recovers plaintext

In [ ]:
from NeuroSync import NeuroSync as NS

# Load our freshly trained cipher
cipher = NS.from_pretrained(SAVE_DIR)
print("Cipher loaded. Ready to encrypt!\n")

# ──────────────────────────────────────────
# 🔒 ENCRYPT — change the message below!
# ──────────────────────────────────────────
message = "Hello NeuroSync"

encrypted = cipher.encrypt(message)
encrypted_readable = NeuroSync.encoding.bits_to_text(encrypted)

print(f"Original:   '{message}'")
print(f"Encrypted:  '{encrypted_readable}'")
print(f"Tensor shape: {encrypted.shape}")
print(f"Tensor (first 20 values): {encrypted[:20].tolist()}")

# ──────────────────────────────────────────
# 🔓 DECRYPT
# ──────────────────────────────────────────
decrypted = cipher.decrypt(encrypted)
match = "YES" if decrypted.rstrip('=').rstrip() == message.ljust(16)[:16].rstrip() else "NO"
print(f"\nDecrypted:  '{decrypted}' {match}")

### Protocol used with sender and receiver

Below is an example of using the `Sender` and `Receiver` classes to implement a simple communication protocol with packetization and acknowledgments. This simulates a more realistic use case where messages are sent in chunks and the receiver sends ACKs back to the sender.

In [ ]:
from NeuroSync import NeuroSync

cipher = NeuroSync.from_pretrained(SAVE_DIR)

# Create the sender and receiver
sender = cipher.create_sender()
receiver = cipher.create_receiver()

# Set the message
msg = "Hello from NeuroSync"

# Sned and receive loop
while msg:
    packets, msg = sender.send(msg)
    for packet in packets:
        message = receiver.receive(packet)
        if message:
            print(f"Received: {message}")
    acks = receiver.get_pending_acks()
    for ack in acks:
        sender.handle_ack(ack)

### Batch Encrypt / Decrypt

Encrypt multiple messages at once — useful for throughput testing.

In [ ]:
# Batch mode through the session API
messages = [
    "hello world     ",
    "test message    ",
    "NeuroSync rocks ",
    "crypto is fun   ",
]

session = cipher.session

ct_batch = session.encrypt_batch(messages)
dec_batch = session.decrypt_batch(ct_batch)

print(f"{'Original':<22} {'Decrypted':<22} {'Match'}")
print("-" * 50)
for orig, dec in zip(messages, dec_batch):
    match = "YES" if orig == dec else "NO"
    print(f"'{orig}'  →  '{dec}'  {match}")

---
## 3️⃣ Code Playground

This is your sandbox. The full NeuroSync API is available — write any code you like.

**Available imports:**
```python
from NeuroSync import NeuroSync, Sender, Receiver, Visualizer
from NeuroSync import NeuroSyncTrainer, TrainingConfig
from NeuroSync import Packet, CryptoSession
from NeuroSync import core, encoding, crypto, security, protocol
```

**Quick reference:**
- `cipher.encrypt(text)` → tensor
- `cipher.decrypt(tensor)` → text
- `cipher.session.encrypt_batch([...])` → tensor
- `cipher.create_sender()` / `cipher.create_receiver()`
- `NeuroSync.encoding.text_to_bits(text)` / `bits_to_text(tensor)`
- `NeuroSync.security.check_total(alice, plain, cipher, key)`
- `NeuroSync.core.Alice`, `Bob`, `Eve` — raw network classes

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  YOUR CODE HERE — experiment freely!                     ║
# ╚══════════════════════════════════════════════════════════╝

# Example: test how many messages decrypt correctly out of 100
from NeuroSync.data.generators import MessageGenerator
from NeuroSync.encoding.batch import text_to_bits_batch, bits_to_text_batch
from NeuroSync.crypto.operations import xor

gen = MessageGenerator(message_length=16)
gen.create_word_list(
    "./neurosync_playground_data/playground_words.txt",
    message_length=16,
    num_words=500
)

test_messages = gen.generate_batch(100)
session = cipher.session

ct = session.encrypt_batch(test_messages)
dec = session.decrypt_batch(ct)

correct = sum(1 for o, d in zip(test_messages, dec) if o == d)
print(f"Accuracy: {correct}/100 = {correct}%")

# Show some examples
print(f"\n{'Original':<22} {'Decrypted':<22} {'OK?'}")
print("-" * 50)
for o, d in zip(test_messages[:10], dec[:10]):
    print(f"'{o}'  →  '{d}'  {'YES' if o == d else 'NO'}")

---
## 4️⃣ Experimental Lab — Tweak Eve & Bob Parameters

This section lets you re-train with different adversarial settings to see how they affect the cipher's security and accuracy. Questions to explore:

- What happens if Eve is much stronger (higher adversarial weight)?
- What if Bob is weaker (smaller hidden size)?
- Does faster key rotation help security?
- Can Eve break a poorly-trained cipher?

In [ ]:
def run_experiment(name, config_overrides, num_episodes=60_000):
    """Run a training experiment with custom overrides and return the result."""
    base = dict(
        training_episodes=num_episodes,
        batch_size=64,
        hidden_size=512,
        num_residual_blocks=2,
        learning_rate=0.001,
        eve_learning_rate=0.002,
        log_interval=2000,
        test_interval=5000,
        scheduler_step_size=8000,
        scheduler_gamma=0.7,
        data_dir="./neurosync_playground_data",
        word_list_file="words.txt",
        word_list_size=4_000,
        maintenance_threshold=98.0,
        maintenance_threshold_exit=88.0,
        consecutive_accuracy_required=2,
    )
    base.update(config_overrides)
    cfg = TrainingConfig(**base)

    print(f"\n{'=' * 60}")
    print(f"  EXPERIMENT: {name}")
    print(f"{'=' * 60}")
    for k, v in config_overrides.items():
        print(f"  {k}: {v}")
    print()

    t = NeuroSyncTrainer(cfg)
    res = t.train()
    return res, t

In [ ]:
# ── Experiment 1: Strong Adversary ──
# High adversarial_max pushes Alice to hide info from Eve harder,
# but risks destabilizing Bob's decryption accuracy.
exp1_result, exp1_trainer = run_experiment(
    "Strong Adversary (adversarial_max=0.4)",
    {"adversarial_max": 0.4, "security_max": 0.2},
    num_episodes=60_000,
)
print(f"\nResult: best_accuracy={exp1_result.best_accuracy:.1f}%")

In [ ]:
# ── Experiment 2: Weak Bob (tiny network) ──
# With only 64 hidden units, Bob struggles — can Eve exploit this?
exp2_result, exp2_trainer = run_experiment(
    "Weak Bob (hidden_size=64, 1 residual block)",
    {"hidden_size": 64, "num_residual_blocks": 1},
    num_episodes=60_000,
)
print(f"\nResult: best_accuracy={exp2_result.best_accuracy:.1f}%")

In [ ]:
# ── Experiment 3: No Adversarial Training ──
# What happens if Eve never pushes Alice to be secure?
exp3_result, exp3_trainer = run_experiment(
    "No Adversary (adversarial_max=0.0)",
    {"adversarial_max": 0.0, "security_max": 0.0},
    num_episodes=60_000,
)
print(f"\nResult: best_accuracy={exp3_result.best_accuracy:.1f}%")

# Now let Eve attack the undefended cipher
print("\n── Eve attacking the undefended cipher ──")
from NeuroSync.encoding.batch import text_to_bits_batch
from NeuroSync.encoding.codec import bits_to_text
from NeuroSync.crypto.operations import xor
from NeuroSync.data.generators import MessageGenerator

alice_exp = exp3_result.alice.eval()
eve_exp = exp3_result.eve.eval()

gen = MessageGenerator(message_length=16)
gen.create_word_list(
    "./neurosync_playground_data/eve_test_words.txt",
    message_length=16,
    num_words=500
)
test_msgs = gen.generate_batch(64)
device = next(alice_exp.parameters()).device
plain_bits = text_to_bits_batch(test_msgs, device=device)

# Eve uses a random key (she doesn't have the real one)
from NeuroSync.encoding.constants import BIT_LENGTH as _BL
eve_key = torch.tensor(
    np.random.choice([-1.0, 1.0], _BL),
    dtype=torch.float32, device=device
).unsqueeze(0).repeat(64, 1)

with torch.no_grad():
    ct = torch.sign(alice_exp(xor(plain_bits, eve_key)))
    eve_dec = eve_exp(ct)
    eve_texts = bits_to_text_batch(eve_dec)

eve_correct = sum(1 for o, d in zip(test_msgs, eve_texts) if o == d)
print(f"Eve's accuracy: {eve_correct}/64 = {100*eve_correct/64:.1f}%")

### Key Rotation Experiment

Test how key rotation affects the protocol — rotate keys more or less frequently.

In [ ]:
from NeuroSync.interface.pair import CommunicationPair
from NeuroSync import NeuroSync as NS

# Use our main trained cipher
cipher_for_kr = NS.from_pretrained(SAVE_DIR)

# Test with aggressive key rotation (every 3 packets)
sender = cipher_for_kr.create_sender()
sender.key_rotation.rotation_interval = 3  # rotate every 3 packets!

receiver = cipher_for_kr.create_receiver()

messages_to_send = ["msg one", "msg two", "msg three", "msg four", "msg five"]
print("Sending with key rotation every 3 packets:\n")

for msg in messages_to_send:
    padded = msg.ljust(16, '=')
    packets, remaining = sender.send(padded)
    for pkt_bytes in packets:
        from NeuroSync.protocol.packet import Packet
        from NeuroSync.protocol.flags import PacketFlags
        pkt = Packet.from_bytes(pkt_bytes)
        if pkt.header.flags & PacketFlags.KEY_CHANGE:
            print(f"  🔑 KEY ROTATION packet (seq={pkt.header.sequence_id})")
            result = receiver.receive(pkt_bytes)
            # Send ACK back so sender commits the new key
            acks = receiver.get_pending_acks()
            for ack in acks:
                sender.handle_ack(ack)
        else:
            result = receiver.receive(pkt_bytes)
            if result is not None:
                match = "YES" if result.strip().rstrip('=') == msg else "NO"
                print(f"  '{msg}' → '{result.strip()}' {match}")

---
## 5️⃣ Under the Hood — Network Weights & Architecture

Peek inside Alice, Bob, and Eve. See the actual learned parameters, temperature values, and architecture details.

In [ ]:
# Load our trained networks
from NeuroSync import NeuroSync as NS

cipher_inspect = NS.from_pretrained(SAVE_DIR)
alice = cipher_inspect.alice
bob = cipher_inspect.bob

print("=" * 60)
print("ALICE (Encoder) Architecture")
print("=" * 60)
print(alice)
print(f"\nLearned temperature: {alice.temperature.item():.4f}")
print(f"Effective temperature (softplus + 0.5): {alice.temp.item():.4f}")

total_params = sum(p.numel() for p in alice.parameters())
trainable = sum(p.numel() for p in alice.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable: {trainable:,}")

print(f"\n{'=' * 60}")
print("BOB (Decoder) Architecture")
print("=" * 60)
print(f"Learned temperature: {bob.temperature.item():.4f}")
print(f"Effective temperature: {bob.temp.item():.4f}")
print(f"Total parameters: {sum(p.numel() for p in bob.parameters()):,}")

In [ ]:
# Detailed weight inspection
print("ALICE — Layer-by-Layer Weight Statistics")
print("=" * 60)
for name, param in alice.named_parameters():
    data = param.data
    print(f"\n  {name}:")
    print(f"    Shape: {list(data.shape)}")
    print(f"    Mean:  {data.mean().item():.6f}")
    print(f"    Std:   {data.std().item():.6f}")
    print(f"    Min:   {data.min().item():.6f}")
    print(f"    Max:   {data.max().item():.6f}")
    print(f"    Zeros: {(data.abs() < 1e-6).sum().item()}")

In [ ]:
# Visualize input projection weights as a text-based heatmap
import numpy as np

w = alice.input_projection.weight.data.cpu().numpy()
print(f"Alice input_projection weight matrix: {w.shape}")
print(f"\nHeatmap of first 16x16 block (scaled to [-1, 1]):")
print()

block = w[:16, :16]
vmin, vmax = block.min(), block.max()
scale = max(abs(vmin), abs(vmax))

chars = " ░▒▓█"
for row in block:
    line = ""
    for val in row:
        normalized = (val / scale + 1) / 2  # 0 to 1
        idx = min(int(normalized * (len(chars) - 1)), len(chars) - 1)
        line += chars[idx] * 2
    print(f"  {line}")
print(f"\n  Legend: ' '=-1.0  '░'=-0.5  '▒'=0.0  '▓'=0.5  '█'=1.0")

### Load Custom Weights

You can provide your own weight tensors to see how the network behaves.

In [ ]:
# Example: inject custom weights into a fresh network and test it
from NeuroSync.core.networks import CryptoNetwork
from NeuroSync.encoding.codec import text_to_bits, bits_to_text
from NeuroSync.crypto.operations import xor

bit_length = playground_config.bit_length
device = next(alice.parameters()).device

custom_net = CryptoNetwork(bit_length, 128, bit_length, name="Custom").to(device)

# Copy Alice's weights as a starting point
custom_net.load_state_dict(alice.state_dict())

# Modify a specific layer — e.g., add noise to the output layer
with torch.no_grad():
    noise = torch.randn_like(custom_net.out.weight) * 0.1
    custom_net.out.weight.add_(noise)

# Test the modified network
key = cipher_inspect.key_manager.to_tensor(1).squeeze(0)
test = "test message    "
bits = torch.tensor(text_to_bits(test), dtype=torch.float32, device=key.device)

with torch.no_grad():
    ct_original = torch.sign(alice(xor(bits, key), single=True))
    ct_modified = torch.sign(custom_net(xor(bits, key), single=True))

    diff = (ct_original != ct_modified).float().mean().item()

print(f"Original Alice ciphertext:  '{bits_to_text(ct_original)}'")
print(f"Modified Alice ciphertext:  '{bits_to_text(ct_modified)}'")
print(f"Bit difference: {diff*100:.1f}%")
print(f"\nBob decrypts original: '{bits_to_text(bob(xor(ct_original, key), single=True))}'")
print(f"Bob decrypts modified: '{bits_to_text(bob(xor(ct_modified, key), single=True))}'")

---
## 6️⃣ Security Analysis Deep Dive

Run the full security analysis suite on the trained cipher. The checks measure:

| Check | What It Tests | Good Value |
|---|---|---|
| **Leakage** | Bit correlation between plaintext & ciphertext | < 0.2 |
| **Diversity** | Variance of ciphertext across different inputs | < 0.2 |
| **Repetition** | Similarity between ciphertext pairs | < 0.2 |
| **Key Sensitivity** | How much ciphertext changes when 1 key bit flips | < 0.2 |

In [ ]:
from NeuroSync.security.analyzer import SecurityAnalyzer
from NeuroSync.security.thresholds import SecurityThresholds
from NeuroSync.encoding.batch import text_to_bits_batch
from NeuroSync.encoding.codec import text_to_bits, bits_to_text
from NeuroSync.crypto.operations import xor
from NeuroSync.data.generators import MessageGenerator

cipher_sec = NS.from_pretrained(SAVE_DIR)
alice_s = cipher_sec.alice.eval()
bob_s = cipher_sec.bob.eval()
device = next(alice_s.parameters()).device

# Generate test data
gen = MessageGenerator(message_length=16)
gen.create_word_list(
    "./neurosync_playground_data/sec_words.txt",
    message_length=16,
    num_words=500
)
test_msgs = gen.generate_batch(64)
plain_bits = text_to_bits_batch(test_msgs, device=device)
key_batch = cipher_sec.key_manager.to_tensor(64)

with torch.no_grad():
    ct = torch.sign(alice_s(xor(plain_bits, key_batch)))

# Run the full analyzer
analyzer = SecurityAnalyzer(SecurityThresholds())
report = analyzer.analyze(alice_s, plain_bits, ct, key_batch)

print("=" * 60)
print("SECURITY ANALYSIS REPORT")
print("=" * 60)
status_emoji = {"ok": "🟢", "warn": "🟡", "bad": "🔴"}
thresholds = SecurityThresholds()

for check_name in ["leakage", "diversity", "repetition", "key_sensitivity"]:
    val = getattr(report, check_name)
    status = thresholds.evaluate_component(val)
    emoji = status_emoji[status.value]
    bar = "█" * int(val * 50) + "░" * (50 - int(val * 50))
    print(f"\n  {check_name.title():20s} {emoji} {val:.4f}")
    print(f"  [{bar}]")

print(f"\n  {'Overall Score':20s} {status_emoji[report.status.value]} {report.overall_score:.4f}")
print(f"  Status: {report.status.value.upper()}")

In [ ]:
# ---------------------------------------------------------------------------
# Image Encryption Demo: Transmit an image through the NeuroSync protocol
# ---------------------------------------------------------------------------
# This cell demonstrates the full Sender/Receiver pipeline on real binary
# data.  An image is encoded to base32, split into 16-character chunks by
# the Sender, encrypted by Alice, packetized with checksums and parity,
# then received and decrypted by Bob on the other end.  Along the way we
# capture the raw encrypted payloads so we can visualise what the cipher-
# text looks like compared to the original and recovered images.
# ---------------------------------------------------------------------------

import base64, io, math
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import matplotlib
import matplotlib.pyplot as plt
from NeuroSync import NeuroSync
from NeuroSync.protocol.packet import Packet
from NeuroSync.protocol.flags import PacketFlags

# -- 1. Create (or upload) a sample image ----------------------------------
# Generate a small but recognisable test image with gradients and shapes.
# To use your own image, replace the block below with:
#     img = Image.open("your_file.png").resize((64, 64))

IMG_SIZE = 64

img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), color=(20, 20, 40))
draw = ImageDraw.Draw(img)
for y in range(IMG_SIZE):
    r = int(255 * y / IMG_SIZE)
    g = int(255 * (IMG_SIZE - y) / IMG_SIZE)
    b = 128
    draw.line([(0, y), (IMG_SIZE - 1, y)], fill=(r, g, b))
draw.rectangle([12, 12, 52, 52], outline=(255, 255, 255), width=2)
draw.ellipse([20, 20, 44, 44], outline=(255, 220, 50), width=2)
draw.line([(0, 0), (IMG_SIZE - 1, IMG_SIZE - 1)], fill=(255, 80, 80), width=1)
draw.line([(IMG_SIZE - 1, 0), (0, IMG_SIZE - 1)], fill=(80, 255, 80), width=1)
try:
    font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 9)
except Exception:
    font = ImageFont.load_default()
draw.text((14, 28), "NS", fill=(255, 255, 255), font=font)

# -- 2. Encode the image to a base32 string --------------------------------
buf = io.BytesIO()
img.save(buf, format="PNG")
image_bytes = buf.getvalue()
b32_string = base64.b32encode(image_bytes).decode("ascii")

print("IMAGE ENCRYPTION ROUND-TRIP DEMO (base32)")
print("=" * 60)
print(f"  Image dimensions:      {IMG_SIZE} x {IMG_SIZE} RGB")
print(f"  Raw PNG size:          {len(image_bytes)} bytes")
print(f"  Base32 string length:  {len(b32_string)} characters")
print(f"  Expected packets:      ~{math.ceil(len(b32_string) / 16)}")
print("=" * 60)
print()

# -- 3. Transmit through the NeuroSync Sender / Receiver protocol ----------
# Uses the same Sender/Receiver pattern as the README example.

cipher = NeuroSync.from_pretrained(SAVE_DIR)
sender = cipher.create_sender()
receiver = cipher.create_receiver()

encrypted_payloads = []          # raw ciphertext bytes (data packets only)
recovered_fragments = []         # decrypted text fragments from the receiver
packets_sent = 0
key_rotations = 0

msg = b32_string
while msg:
    packets, msg = sender.send(msg)
    for pkt_bytes in packets:
        pkt = Packet.from_bytes(pkt_bytes)

        if pkt.header.flags.has(PacketFlags.KEY_CHANGE):
            key_rotations += 1
            receiver.receive(pkt_bytes)
            for ack in receiver.get_pending_acks():
                sender.handle_ack(ack)
        else:
            packets_sent += 1
            encrypted_payloads.append(pkt.payload)
            result = receiver.receive(pkt_bytes)
            if result is not None:
                recovered_fragments.append(result)

    for ack in receiver.get_pending_acks():
        sender.handle_ack(ack)

recovered_b32 = "".join(recovered_fragments)

# The receiver strips trailing '=' padding which may remove valid base32
# padding characters.  Restore them so the length is a multiple of 8.
pad_needed = len(recovered_b32) % 8
if pad_needed:
    recovered_b32 += "=" * (8 - pad_needed)

print(f"  Packets transmitted:   {packets_sent}")
print(f"  Key rotations:         {key_rotations}")
print(f"  Recovered base32:      {len(recovered_b32)} / {len(b32_string)} chars")
print()

# -- 4. Build the encrypted-payload visualisation --------------------------
# Each data packet payload is a serialised float32 tensor of +/-1 values
# produced by Alice.  Concatenate them and map to pixel intensities so the
# viewer can see what the ciphertext looks like: pure noise.

raw_encrypted = b"".join(encrypted_payloads)
enc_floats = np.frombuffer(raw_encrypted, dtype=np.float32)

enc_pixels = ((enc_floats + 1.0) / 2.0 * 255.0).clip(0, 255).astype(np.uint8)

side = int(math.ceil(math.sqrt(len(enc_pixels))))
padded = np.zeros(side * side, dtype=np.uint8)
padded[: len(enc_pixels)] = enc_pixels
enc_image = padded.reshape(side, side)

# -- 5. Recover the image from the decrypted base32 -----------------------
try:
    recovered_bytes = base64.b32decode(recovered_b32)
    recovered_img = Image.open(io.BytesIO(recovered_bytes))
    recovery_ok = True
except Exception:
    recovered_img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (40, 0, 0))
    draw_err = ImageDraw.Draw(recovered_img)
    draw_err.text((6, 24), "DECODE ERR", fill=(200, 80, 80))
    recovery_ok = False

# -- 6. Display: Original | Encrypted Noise | Recovered -------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 5), facecolor="#f8f8f8")
fig.suptitle(
    "NeuroSync Protocol  --  Image Encryption Round-Trip (base32)",
    fontsize=14, fontweight="bold", y=0.98,
)

axes[0].imshow(np.array(img))
axes[0].set_title("Original", fontsize=12, fontweight="bold")
axes[0].set_xlabel(f"{IMG_SIZE}x{IMG_SIZE}  |  {len(image_bytes)} bytes PNG")
axes[0].set_xticks([])
axes[0].set_yticks([])
axes[0].spines[:].set_color("#cccccc")

axes[1].imshow(enc_image, cmap="inferno", interpolation="nearest")
axes[1].set_title("Encrypted Payloads (ciphertext)", fontsize=12, fontweight="bold")
axes[1].set_xlabel(f"{side}x{side}  |  {len(enc_floats)} float32 values from Alice")
axes[1].set_xticks([])
axes[1].set_yticks([])
axes[1].spines[:].set_color("#cccccc")

axes[2].imshow(np.array(recovered_img))
tag = "RECOVERED" if recovery_ok else "FAILED"
axes[2].set_title(f"Decrypted ({tag})", fontsize=12, fontweight="bold")
if recovery_ok:
    matching = sum(a == b for a, b in zip(b32_string, recovered_b32))
    total = max(len(b32_string), len(recovered_b32))
    axes[2].set_xlabel(f"char accuracy: {matching}/{total} ({100*matching/total:.1f}%)")
else:
    axes[2].set_xlabel("base32 decode failed (model needs more training)")
axes[2].set_xticks([])
axes[2].set_yticks([])
axes[2].spines[:].set_color("#cccccc")

plt.tight_layout()
plt.show()

# -- 7. Transmission summary ----------------------------------------------
print()
print("=" * 60)
print("TRANSMISSION SUMMARY")
print("=" * 60)
print(f"  Base32 chars sent:       {len(b32_string)}")
print(f"  Base32 chars recovered:  {len(recovered_b32)}")
print(f"  Data packets:            {packets_sent}")
print(f"  Key rotations:           {key_rotations}")
if recovery_ok and len(b32_string) > 0:
    matching = sum(a == b for a, b in zip(b32_string, recovered_b32))
    total = max(len(b32_string), len(recovered_b32))
    accuracy = 100.0 * matching / total
    print(f"  Character accuracy:      {matching}/{total} ({accuracy:.1f}%)")
    if recovered_b32 == b32_string:
        print(f"  Result:                  PERFECT -- image recovered exactly")
    else:
        print(f"  Result:                  PARTIAL -- {total - matching} chars differ")
        print(f"                           (train longer for higher accuracy)")
else:
    print(f"  Result:                  FAILED -- base32 could not be decoded")
    print(f"                           (the model needs more training epochs)")
print("=" * 60)

---
## 🎓 What You've Learned

1. **Neural networks can learn encryption** — without being given any traditional crypto algorithm
2. **Adversarial training drives security** — Eve's pressure forces Alice to produce harder-to-crack ciphertext
3. **It's a balancing act** — too much adversarial pressure and Bob loses accuracy; too little and the cipher is weak
4. **The protocol stack** handles real-world concerns like packetization, error correction, and key rotation

### Next Steps

- Train with the **full default config** on a GPU for production-grade results
- Explore the [NeuroSync GitHub repo](https://github.com/CooDiiNgg/NeuroSync) for more examples
- Try integrating with the **Sender/Receiver protocol** for network communication
- Read the [original Google Brain paper](https://arxiv.org/abs/1610.06918) that inspired this project

```bash
pip install NeuroSync
```
